<a href="https://colab.research.google.com/github/tatyanalitvin/python_for_ds_tasks/blob/dev/Unit_07/HW_RecSys_Goodbooks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнє завдання: Рекомендаційні системи на реальних даних (Goodbooks-10k)

У цьому завданні Ви реалізуєте сучасні (advanced) архітектури рекомендаційних систем із фінального блоку лекції — але вже **не на іграшкових даних, а на реальному датасеті книжкових рейтингів Goodbooks-10k** (десятки тисяч користувачів, тисячі книг, мільйони оцінок).

Це дасть Вам змогу побачити, як підходи поводяться, коли даних справді багато: чому контентних ознак буває замало, як працює retrieval на тисячах елементів, і чому офлайн-метрики на кшталт Recall@K не такі високі, як хотілося б.

**Архітектури, які Ви зберете:** Vector Space Model, Two-Tower, Concat-based ranking (NCF) та двоетапний пайплайн Retrieval → Ranking.

**Стек:** `numpy`, `pandas`, `scikit-learn`, `torch`. GPU не обов'язковий, але з ним тренування буде швидшим (у Colab: *Runtime → Change runtime type → GPU*).

---

## Про датасет

[Goodbooks-10k](https://www.kaggle.com/datasets/zygmunt/goodbooks-10k) — це ~6 млн оцінок 10 000 найпопулярніших книг від 53 424 користувачів. Складається з кількох файлів:

- `ratings.csv` — оцінки: `user_id, book_id, rating` (1–5);
- `books.csv` — метадані книг: `book_id, goodreads_book_id, authors, title, average_rating, ...`;
- `book_tags.csv` — теги/полиці, які користувачі вішали на книги: `goodreads_book_id, tag_id, count`;
- `tags.csv` — розшифровка тегів: `tag_id, tag_name`.

**Важливий нюанс:** на відміну від навчального прикладу, тут **немає готових жанрів**. Жанри доведеться сконструювати самостійно з користувацьких тегів — а це шумні дані (юзери можуть зазначати що завгодно). Це реалістична задача feature engineering, і ми її розберемо в підготовчій частині.

Ще один нюанс із реальних даних: `book_tags.csv` посилається на `goodreads_book_id`, а `ratings.csv` — на `book_id`. Щоб їх поєднати, потрібен джойн через `books.csv`.


## Крок 0. Завантаження даних

Є три способи дістати дані — оберіть будь-який.

**Спосіб A — Kaggle API (рекомендований).** Завантаження з Kaggle API. Зручно, бо декілька файлів і вони завантажаться всі самостійно. Для цього способу завантажте свій `kaggle.json` (Kaggle → Account → Create New API Token), потім виконайте:
```python
from google.colab import files; files.upload()   # оберіть kaggle.json
```
і розкоментуйте відповідний блок нижче.

**Спосіб B — ручне завантаження.** Завантажте архів з посилання на датасет вище з Kaggle, розпакуйте і покладіть `ratings.csv`, `books.csv`, `book_tags.csv`, `tags.csv` поруч із ноутбуком (або через панель Files у Colab).

**Спосіб C — GitHub-дзеркало (фолбек).** Оригінальний автор виклав файли і на GitHub — код нижче підхопить їх автоматично, якщо локально файлів немає.


In [1]:
# (Спосіб A) Kaggle API — розкоментуйте, якщо завантажили kaggle.json
# !pip -q install kaggle
# import os, shutil
# os.makedirs("/root/.kaggle", exist_ok=True)
# shutil.move("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
# !kaggle datasets download -d zygmunt/goodbooks-10k --unzip -p .

In [2]:
import os
import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master"
FILES = ["ratings.csv", "books.csv", "book_tags.csv", "tags.csv"]

def load(fname):
    """Спочатку шукаємо файл локально, інакше тягнемо з GitHub-дзеркала."""
    if os.path.exists(fname):
        return pd.read_csv(fname)
    print(f"{fname} не знайдено локально — завантажую з GitHub...")
    return pd.read_csv(f"{GITHUB}/{fname}")

ratings = load("ratings.csv")
books = load("books.csv")
book_tags = load("book_tags.csv")
tags = load("tags.csv")

print("ratings:", ratings.shape)
print("books:  ", books.shape)
print("book_tags:", book_tags.shape, "| tags:", tags.shape)
books[["book_id", "authors", "title", "average_rating"]].head()

ratings.csv не знайдено локально — завантажую з GitHub...
books.csv не знайдено локально — завантажую з GitHub...
book_tags.csv не знайдено локально — завантажую з GitHub...
tags.csv не знайдено локально — завантажую з GitHub...
ratings: (5976479, 3)
books:   (10000, 23)
book_tags: (999912, 3) | tags: (34252, 2)


,book_id,authors,title,average_rating
0,1,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",4.34
1,2,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,4.44
2,3,Stephenie Meyer,"Twilight (Twilight, #1)",3.57
3,4,Harper Lee,To Kill a Mockingbird,4.25
4,5,F. Scott Fitzgerald,The Great Gatsby,3.89


## Крок 1. Інженерія жанрів із тегів (feature engineering)

Жанрів у датасеті немає, але є користувацькі теги. Виберемо набір канонічних жанрів і для кожної книги позначимо, які з них їй приписали користувачі. Так ми отримаємо **бінарну матрицю book × genre** — це й будуть контентні ознаки айтемів (аналог `movie_feats_df` із лекції, але здобутий з реальних шумних даних).


In [3]:
# Канонічні жанри, які шукаємо серед тегів
GENRES = ["fantasy", "romance", "mystery", "thriller", "horror", "historical",
          "science-fiction", "young-adult", "nonfiction", "classics",
          "contemporary", "crime"]

# tag_name -> tag_id
name_to_tagid = dict(zip(tags["tag_name"], tags["tag_id"]))
genre_tag_ids = {g: name_to_tagid[g] for g in GENRES if g in name_to_tagid}

# book_tags використовує goodreads_book_id -> мапимо у book_id через books.csv
gid_to_bid = dict(zip(books["goodreads_book_id"], books["book_id"]))
tagid_to_genre = {tid: g for g, tid in genre_tag_ids.items()}

bt = book_tags[book_tags["tag_id"].isin(genre_tag_ids.values())].copy()
bt["book_id"] = bt["goodreads_book_id"].map(gid_to_bid)
bt = bt.dropna(subset=["book_id"])
bt["genre"] = bt["tag_id"].map(tagid_to_genre)

# бінарна матриця book × genre (жанр присутній, якщо користувачі його тегали)
genre_matrix = (
    bt.pivot_table(index="book_id", columns="genre", values="count", aggfunc="sum", fill_value=0)
      .reindex(columns=GENRES, fill_value=0) > 0
).astype(int)

print("Книг із хоча б одним жанром:", (genre_matrix.sum(axis=1) > 0).sum(), "/", len(books))
print("\nРозподіл жанрів:")
print(genre_matrix.sum().sort_values(ascending=False))
genre_matrix.head()

Книг із хоча б одним жанром: 9954 / 10000

Розподіл жанрів:
genre
contemporary       5287
fantasy            4259
romance            4251
mystery            3686
young-adult        3630
classics           2785
historical         2544
thriller           2522
science-fiction    2222
crime              2083
nonfiction         1833
horror             1372
dtype: int64


genre,fantasy,romance,mystery,thriller,horror,historical,science-fiction,young-adult,nonfiction,classics,contemporary,crime
book_id,,,,,,,,,,,,
1,1,1,0,1,0,0,1,1,0,0,1,0
2,1,0,1,0,0,0,0,1,0,1,1,0
3,1,0,0,0,1,0,1,1,0,0,1,0
4,0,0,1,0,0,1,0,1,0,1,1,1
5,0,1,0,0,0,1,0,1,0,1,0,0


## Крок 2. Підвибірка під Colab

6 млн рейтингів — забагато для навчального ноутбука на CPU. Візьмемо **топ-N найпопулярніших книг** і **активних користувачів** (хто поставив ≥ 20 оцінок), а тоді обмежимо число користувачів. Так зберігається щільність взаємодій, а тренування лишається швидким.

> Якщо у Вас GPU або багато часу — сміливо збільшуйте `TOP_BOOKS` та `N_USERS`.


In [4]:
TOP_BOOKS = 1500       # скільки найпопулярніших книг лишити
MIN_USER_RATINGS = 20  # мінімум оцінок на користувача
N_USERS = 2000         # скільки користувачів узяти у підвибірку
LIKE_THRESHOLD = 4     # rating >= 4 вважаємо "лайком" (позитивна взаємодія)

rng = np.random.RandomState(42)

top_books = ratings["book_id"].value_counts().head(TOP_BOOKS).index
r = ratings[ratings["book_id"].isin(top_books)]
active = r["user_id"].value_counts()
r = r[r["user_id"].isin(active[active >= MIN_USER_RATINGS].index)]
sample_users = rng.choice(r["user_id"].unique(), size=min(N_USERS, r["user_id"].nunique()), replace=False)
r = r[r["user_id"].isin(sample_users)].copy()

# лишаємо тільки книги, для яких є жанрові ознаки
r = r[r["book_id"].isin(genre_matrix.index)].copy()

items = sorted(r["book_id"].unique())
users = sorted(r["user_id"].unique())
genre_matrix = genre_matrix.reindex(items).fillna(0).astype(int)

print(f"Взаємодій: {len(r):,} | користувачів: {len(users):,} | книг: {len(items):,}")
print(f"Щільність: {len(r) / (len(users) * len(items)):.4f}")

Взаємодій: 140,934 | користувачів: 2,000 | книг: 1,496
Щільність: 0.0471


In [5]:
import torch
import torch.nn as nn

torch.manual_seed(42)

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {b: i for i, b in enumerate(items)}
title_of = dict(zip(books["book_id"], books["title"]))

item_feats = torch.tensor(genre_matrix.values, dtype=torch.float32)  # (M, n_genres)
M = len(items)
n_genres = item_feats.shape[1]

# train/val split по взаємодіях
r = r.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(r) * 0.2)
val_df = r.iloc[:n_val]
train_df = r.iloc[n_val:]

# позитивні пари (лайки) у train
train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD]
pos_u = torch.tensor([user_to_idx[u] for u in train_pos["user_id"]])
pos_i = torch.tensor([item_to_idx[b] for b in train_pos["book_id"]])

# що користувач уже бачив (щоб не рекомендувати повторно і не семплити як негатив)
from collections import defaultdict
seen_by_user = defaultdict(set)
for u, b in zip(train_df["user_id"], train_df["book_id"]):
    seen_by_user[user_to_idx[u]].add(item_to_idx[b])

# val-лайки для оцінки якості
val_pos = defaultdict(set)
for row in val_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        val_pos[user_to_idx[row.user_id]].add(item_to_idx[row.book_id])

print(f"Позитивних пар у train: {len(pos_u):,} | користувачів з val-лайками: {len(val_pos):,}")

Позитивних пар у train: 77,070 | користувачів з val-лайками: 1,991


## Крок 3. Метрика оцінки якості рангування

В лекції ми з вами для оцінки якості використовували **RMSE**. Це валідний варіант, коли треба швидко оцінити якість рек. моделі, але спрощений. RMSE показує, наскільки точно модель передбачає оцінку, яку користувач поставить елементу.

В реальних системах нас ще цікавить **якість ранжування** — наскільки релевантні елементи потрапили в топ списку, який ми реально показуємо користувачу. Для цього використовують ранжувальні метрики: **Precision@K**, **Recall@K**, **NDCG**, **MAP**, **MRR**.

Детальніше можна познайомитись з цими мериками тут:
- огляд метрик для рекомендаційних систем: https://www.evidentlyai.com/ranking-metrics/evaluating-recommender-systems
- Precision та Recall at K: https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k

Нижче давайте реалізуємо функцію `recall_at_k` і будемо оцінювати нею всі наші моделі.

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b2e_6577812c4d677925f1ab5f84_precision_recall_k9.png)

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b47_657781b1f9c868e0cda088f6_precision_recall_k11.png)

**Як працює `recall_at_k`:**

1. Для кожного користувача ми беремо його реальні вподобання з валідаційної вибірки (`val_pos` — книги, які він оцінив на ≥ 4), просимо модель оцінити всі книги й відбираємо топ-K рекомендацій. Перед цим прибираємо книги, які користувач уже бачив у train (щоб не рекомендувати відоме).

2. Далі рахуємо, скільки книг із топ-K справді потрапили в його вподобання (`hits`), і ділимо на загальну кількість релевантних книг (обмежену K, бо більше за K у топ і не влізе).

3. Усереднюємо по всіх користувачах — і отримуємо одне число від 0 до 1: **яку частку того, що користувачу реально сподобалось, модель змогла підняти в топ-K.**

In [6]:
def recall_at_k(score_fn, k=10):
    """Частка val-лайків, що потрапили у топ-k рекомендацій (усереднена по користувачах).
    score_fn(user_idx_tensor) -> матриця оцінок (n_users, M)."""
    eval_users = list(val_pos.keys())
    hits, total = 0, 0
    with torch.no_grad():
        scores = score_fn(torch.tensor(eval_users))  # (len(eval_users), M)
        for row, u in enumerate(eval_users):
            s = scores[row].clone()
            for i in seen_by_user[u]:
                s[i] = -1e9  # прибираємо вже побачене
            topk = torch.topk(s, k).indices.tolist()
            truth = val_pos[u]
            hits += len(set(topk) & truth)
            total += min(len(truth), k)
    return hits / max(total, 1)

---
## Завдання 1. Vector Space Model (векторний підхід)

Перетворимо і книги, і користувачів на вектори в спільному просторі та шукатимемо рекомендації через cosine similarity. Роль ембединга книги відіграє її **нормалізований вектор жанрів** (пояснення про нормалізацію - нижче), а вектор користувача збираємо як **average pooling** ембедингів книг, які він уподобав.

**Що зробити:**

1. Побудуйте `item_emb` — матрицю L2-нормалізованих жанрових векторів усіх книг.
2. Реалізуйте функцію `user_vector(user_idx)` — зважене (за оцінкою) середнє ембедингів уподобаних книг користувача.
3. Реалізуйте функцію `vsm_scores(user_idxs)` — оцінки (cosine) усіх книг для набору користувачів, та порахуйте `recall_at_k`.
4. Покажіть топ-5 рекомендацій для одного користувача (з назвами книг).

**Довідка:**

L2-нормалізація — це ділення вектора на його довжину (L2-норму), щоб отримати вектор тієї ж напрямленості, але одиничної довжини.

Норма рахується як корінь із суми квадратів компонент:

$$\|v\|_2 = \sqrt{(v_1^2 + v_2^2 + \dots + v_n^2)}$$

а сам нормалізований вектор — це
$$\hat{v} = \frac{v}{\|v\|_2}$$

Навіщо це в рекомендаційних системах: після нормалізації **косинусна подібність зводиться до простого скалярного добутку**. Бо $\cos(a, b) = \frac{a \cdot b}{\|a\|\|b\|}$, і якщо обидва вектори вже одиничної довжини, знаменник = 1, тож $\cos(a,b) = a \cdot b$. Це і швидше, і прибирає вплив «довжини» вектора — порівнюється лише напрямок (тобто склад жанрів/смаків), а не те, скільки книг користувач оцінив.

*Приклад:*

Вектор `[3, 4]` має довжину $\sqrt{(9+16)}=5$, після нормалізації стає `[0.6, 0.8]` — напрямок той самий, довжина 1.

In [7]:
# матриця L2-нормалізованих жанрових векторів усіх книг.
item_emb = item_feats / item_feats.norm(dim=1, keepdim=True).clamp(min=1e-8)  # (M, n_genres)
item_emb


tensor([[0.4082, 0.4082, 0.0000,  ..., 0.0000, 0.4082, 0.0000],
        [0.4472, 0.0000, 0.4472,  ..., 0.4472, 0.4472, 0.0000],
        [0.4472, 0.0000, 0.0000,  ..., 0.0000, 0.4472, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.7071, 0.0000, 0.0000],
        [0.5774, 0.0000, 0.0000,  ..., 0.5774, 0.0000, 0.0000],
        [0.4082, 0.0000, 0.4082,  ..., 0.0000, 0.4082, 0.4082]])

In [8]:
# зважене (за оцінкою) середнє
liked_by_user = defaultdict(list)  # user_idx -> [(item_idx, rating), ...]
for row in train_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        liked_by_user[user_to_idx[row.user_id]].append((item_to_idx[row.book_id], row.rating))

In [9]:
# зважене (за оцінкою) середнє
def user_vector(user_idx):
    liked = liked_by_user.get(user_idx, [])
    if not liked:
        return torch.zeros(n_genres)
    idxs = torch.tensor([i for i, _ in liked])
    w = torch.tensor([rt for _, rt in liked], dtype=torch.float32).unsqueeze(1)  # (k, 1)
    vec = (item_emb[idxs] * w).sum(0) / w.sum() # avg pooling з вагами
    return vec / vec.norm().clamp(min=1e-8) # нормалізуємо -> cos == dot


In [10]:
# оцінки (cosine) усіх книг для набору користувачів, та порахуйте recall_at_k.
def vsm_scores(user_idxs):
    U = torch.stack([user_vector(int(u)) for u in user_idxs])  # (n_users, n_genres)
    return U @ item_emb.t()                                    # (n_users, M)

In [11]:
print(f"VSM Recall@10: {recall_at_k(vsm_scores, k=10):.4f}")

VSM Recall@10: 0.0522


In [12]:
# Топ-5 рекомендацій для одного користувача
def show_recommendations(score_fn, user_idx, k=5):
    with torch.no_grad():
        s = score_fn(torch.tensor([user_idx]))[0].clone()
    for i in seen_by_user[user_idx]:
        s[i] = -1e9  # прибираємо вже побачене
    topk = torch.topk(s, k).indices.tolist()
    print(f"\nТоп-{k} рекомендацій для користувача idx={user_idx}:")
    for rank, i in enumerate(topk, 1):
        print(f"  {rank}. {title_of.get(items[i], '?')}")

example_user = list(val_pos.keys())[0]
show_recommendations(vsm_scores, example_user, k=5)



Топ-5 рекомендацій для користувача idx=316:
  1. The Battle of the Labyrinth (Percy Jackson and the Olympians, #4)
  2. Hush, Hush (Hush, Hush, #1)
  3. Fallen (Fallen, #1)
  4. Shadow Kiss (Vampire Academy, #3)
  5. Beautiful Creatures (Caster Chronicles, #1)


**Питання:** Recall@10 у векторного підходу досить низький. Чому?


- Модель спирається лиш на 12 жанрів, для книги то виглядає як oversimplification.

---
## Завдання 2. Two-Tower архітектура

У Завданні 1 вектор користувача рахувався «вручну». Two-Tower натомість **навчає дві окремі башти**: User Tower (з ембединга user_id) та Item Tower (з жанрових ознак). Мережа зводить вектори уподобаних пар близько, а випадкових — далеко. Перевага: вектори книг рахуються один раз і кладуться в індекс (наприклад, FAISS) для швидкого retrieval — рахувати в реальному часі треба лише вектор користувача. Це **late fusion**.

**Що зробити:**

1. Реалізуйте `TwoTower` (user_tower через `nn.Embedding`, item_tower зі жанрових ознак), виходи L2-нормалізуйте.
2. Навчіть на лайках як позитивах і **negative sampling з усього корпусу** (як у пейпері від YouTube) з `BCEWithLogitsLoss` - він є реалізований в PyTorch.
3. Порахуйте `recall_at_k` через попередньо обчислені вектори книг і покажіть приклад рекомендацій.

> **Підказка.** Множте логіти на «температуру» (\~10), бо скалярний добуток нормалізованих векторів лежить у [-1, 1].
> Множення на температуру (\~10) розтягує діапазон логітів до [-10, 10], і тоді сигмоїда може видавати по-справжньому впевнені ймовірності (близькі до 0 і 1). Це дає лосу нормальний градієнт і модель навчається.


In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"
item_feats_dev = item_feats.to(device)
pos_u_dev, pos_i_dev = pos_u.to(device), pos_i.to(device)
n_pos = len(pos_u_dev)

In [14]:
# Two-Tower = User Tower (nn.Embedding) + Item Tower (зі жанрів)
class TwoTower(nn.Module):
    def __init__(self, n_users, n_genres, dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_tower = nn.Sequential(
            nn.Linear(n_genres, 64),
            nn.ReLU(),
            nn.Linear(64, dim),
        )

    @staticmethod
    def _l2(v):
        return v / v.norm(dim=1, keepdim=True).clamp(min=1e-8)

    def user_forward(self, u):
        return self._l2(self.user_emb(u))

    def item_forward(self, feats):
        return self._l2(self.item_tower(feats))

    def forward(self, u, feats):
        # скалярний добуток нормалізованих веж == cosine (late fusion)
        return (self.user_forward(u) * self.item_forward(feats)).sum(1)


In [15]:
torch.manual_seed(42)
tt = TwoTower(len(users), n_genres, dim=32).to(device)
opt = torch.optim.Adam(tt.parameters(), lr=1e-2)
bce = nn.BCEWithLogitsLoss()

In [16]:
EPOCHS, BATCH, NEG, TEMP = 15, 1024, 4, 10.0  # TEMP розтягує логіти з [-1,1] до [-10,10]


In [17]:
for epoch in range(EPOCHS):
    perm = torch.randperm(n_pos, device=device)
    running = 0.0
    for start in range(0, n_pos, BATCH):
        b = perm[start:start + BATCH]
        u, i = pos_u_dev[b], pos_i_dev[b]
        # negative sampling з усього корпусу (як у YouTube-пейпері)
        neg_i = torch.randint(0, M, (len(b) * NEG,), device=device)
        neg_u = u.repeat_interleave(NEG)
        all_u = torch.cat([u, neg_u])
        all_i = torch.cat([i, neg_i])
        labels = torch.cat([torch.ones(len(b), device=device),
                            torch.zeros(len(b) * NEG, device=device)])
        logits = tt(all_u, item_feats_dev[all_i]) * TEMP
        loss = bce(logits, labels)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item() * len(b)
    print(f"epoch {epoch + 1:2d}/{EPOCHS}  loss={running / n_pos:.4f}")


epoch  1/15  loss=0.6665
epoch  2/15  loss=0.4996
epoch  3/15  loss=0.4858
epoch  4/15  loss=0.4658
epoch  5/15  loss=0.4458
epoch  6/15  loss=0.4305
epoch  7/15  loss=0.4206
epoch  8/15  loss=0.4134
epoch  9/15  loss=0.4086
epoch 10/15  loss=0.4039
epoch 11/15  loss=0.3996
epoch 12/15  loss=0.3974
epoch 13/15  loss=0.3943
epoch 14/15  loss=0.3910
epoch 15/15  loss=0.3886


In [18]:
# Recall через попередньо обчислені вектори книг
@torch.no_grad()
def tt_scores(user_idxs):
    item_vecs = tt.item_forward(item_feats_dev)        # рахуємо один раз
    u = tt.user_forward(user_idxs.to(device))
    return (u @ item_vecs.t()).cpu()                    # (n_users, M)

print(f"\nTwo-Tower Recall@10: {recall_at_k(tt_scores, k=10):.4f}")
show_recommendations(tt_scores, example_user, k=5)



Two-Tower Recall@10: 0.0857

Топ-5 рекомендацій для користувача idx=316:
  1. Catching Fire (The Hunger Games, #2)
  2. The Hunger Games (The Hunger Games, #1)
  3. The Host (The Host, #1)
  4. The Hunger Games Trilogy Boxset (The Hunger Games, #1-3)
  5. Inkheart (Inkworld, #1)


---
## Завдання 3. Concat-based ranking (NCF)

На відміну від Two-Tower (late fusion), тут **early fusion**: склеюємо ембединг користувача і ознаки книги в один вектор і пропускаємо через MLP, який сам моделює крос-взаємодії. Платою є те, що модель **не можна заіндексувати** — щоб знайти найкращу книгу, треба прогнати кожну пару (user, item). Тому її використовують лише на фінальному ранжуванні кількох кандидатів.

**Що зробити:**

1. Реалізуйте `NCF`: `concat(user_embedding, item_genre_features)` → MLP → один логіт.
2. Навчіть на тих самих позитивах/негативах.
3. Реалізуйте `rank_ncf(user_idx, candidate_idxs)` — ранжування заданого списку кандидатів за `sigmoid` логіта.


In [19]:
# NCF: concat(user_embedding, item_genre_features) -> MLP -> один логіт (early fusion)
class NCF(nn.Module):

    def __init__(self, n_users, n_genres, dim=32, hidden=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim + n_genres, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, u, feats):
        x = torch.cat([self.user_emb(u), feats], dim=1)  # склеюємо одразу -> MLP моделює крос-взаємодії
        return self.mlp(x).squeeze(1)

In [20]:
torch.manual_seed(42)
ncf = NCF(len(users), n_genres, dim=32).to(device)
opt = torch.optim.Adam(ncf.parameters(), lr=1e-3)
bce = nn.BCEWithLogitsLoss()

In [21]:
EPOCHS, BATCH, NEG = 15, 1024, 4

for epoch in range(EPOCHS):
    perm = torch.randperm(n_pos, device=device)
    running = 0.0
    for start in range(0, n_pos, BATCH):
        b = perm[start:start + BATCH]
        u, i = pos_u_dev[b], pos_i_dev[b]
        neg_i = torch.randint(0, M, (len(b) * NEG,), device=device)
        neg_u = u.repeat_interleave(NEG)
        all_u = torch.cat([u, neg_u])
        all_i = torch.cat([i, neg_i])
        labels = torch.cat([torch.ones(len(b), device=device),
                            torch.zeros(len(b) * NEG, device=device)])
        logits = ncf(all_u, item_feats_dev[all_i]) # MLP -> логіти не обмежені, температура не потрібна
        loss = bce(logits, labels)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item() * len(b)
    print(f"epoch {epoch + 1:2d}/{EPOCHS}  loss={running / n_pos:.4f}")


epoch  1/15  loss=0.5262
epoch  2/15  loss=0.4986
epoch  3/15  loss=0.4951
epoch  4/15  loss=0.4925
epoch  5/15  loss=0.4898
epoch  6/15  loss=0.4867
epoch  7/15  loss=0.4829
epoch  8/15  loss=0.4784
epoch  9/15  loss=0.4726
epoch 10/15  loss=0.4668
epoch 11/15  loss=0.4607
epoch 12/15  loss=0.4549
epoch 13/15  loss=0.4494
epoch 14/15  loss=0.4446
epoch 15/15  loss=0.4404


In [22]:
# rank_ncf: ранжування заданого списку кандидатів за sigmoid(логіта)
@torch.no_grad()
def rank_ncf(user_idx, candidate_idxs):
    cand = torch.tensor(candidate_idxs, device=device)
    u = torch.full((len(cand),), int(user_idx), device=device)
    scores = torch.sigmoid(ncf(u, item_feats_dev[cand])).cpu()
    order = torch.argsort(scores, descending=True)
    ranked = [candidate_idxs[i] for i in order.tolist()]
    return ranked, scores[order].tolist()

# для повноти: Recall@10, якщо NCF ранжує одразу всі книги (score_fn для recall_at_k)
@torch.no_grad()
def ncf_scores(user_idxs):
    item_all = item_feats_dev
    rows = []
    for u in user_idxs:
        uu = torch.full((M,), int(u), device=device)
        rows.append(torch.sigmoid(ncf(uu, item_all)).cpu())
    return torch.stack(rows)

In [23]:
print(f"\nNCF Recall@10 (ранжує всі книги): {recall_at_k(ncf_scores, k=10):.4f}")


NCF Recall@10 (ранжує всі книги): 0.0415


---
## Завдання 4. Двоетапний пайплайн Retrieval → Ranking

Поєднаємо все так, як це працює у великих системах: **Two-Tower швидко відбирає кандидатів** (retrieval серед усіх книг), а **NCF точно ранжує** цю коротку добірку.

**Що зробити:**

1. `retrieve(user_idx, n_candidates)` — топ-N книг за Two-Tower (Завдання 2), без уже побачених.
2. `recommend_pipeline(user_idx, n_candidates, top_k)` — прогнати кандидатів через `rank_ncf` (Завдання 3).
3. Показати для кількох користувачів: що відібрав retrieval і що залишив ranking.


In [24]:
# кешуємо вектори книг один раз
# Two-Tower retrieval не залежить від користувача
with torch.no_grad():
    ITEM_VECS = tt.item_forward(item_feats_dev) # (M, dim)


In [25]:
# топ-N книг за Two-Tower, без уже побачених
@torch.no_grad()
def retrieve(user_idx, n_candidates=100):
    u = tt.user_forward(torch.tensor([user_idx], device=device))
    s = (u @ ITEM_VECS.t())[0].clone()                 # (M,)
    for i in seen_by_user[user_idx]:
        s[i] = -1e9
    return torch.topk(s, n_candidates).indices.tolist()

In [26]:
def recommend_pipeline(user_idx, n_candidates=100, top_k=10):
    candidates = retrieve(user_idx, n_candidates) # швидкий відбір серед усіх книг
    ranked, _ = rank_ncf(user_idx, candidates) # ранжування короткої добірки
    return ranked[:top_k]

In [27]:
# Recall@10 усього пайплайна (retrieval=200 -> ranking)
def pipeline_scores(user_idxs, n_candidates=200):
    scores = torch.full((len(user_idxs), M), -1e9)
    for row, u in enumerate(user_idxs):
        u = int(u)
        cands = retrieve(u, n_candidates)
        ranked, sc = rank_ncf(u, cands)
        for c, val in zip(ranked, sc):
            scores[row, c] = val
    return scores

In [28]:
print(f"Pipeline Recall@10 (retrieval=200 -> NCF ranking): {recall_at_k(pipeline_scores, k=10):.4f}\n")

Pipeline Recall@10 (retrieval=200 -> NCF ranking): 0.0546



In [29]:
# Для кількох користувачів
for u in list(val_pos.keys())[:3]:
    cands = retrieve(u, n_candidates=10)               # топ-10 кандидатів від retrieval
    final = recommend_pipeline(u, n_candidates=100, top_k=5)
    print(f"Користувач idx={u}")
    print("  Retrieval (Two-Tower, топ-10 кандидатів):")
    for i in cands:
        print(f"     - {title_of.get(items[i], '?')}")
    print("  Ranking (NCF переранжував, фінальний топ-5):")
    for i in final:
        print(f"     * {title_of.get(items[i], '?')}")
    print()


Користувач idx=316
  Retrieval (Two-Tower, топ-10 кандидатів):
     - Catching Fire (The Hunger Games, #2)
     - The Host (The Host, #1)
     - The Hunger Games Trilogy Boxset (The Hunger Games, #1-3)
     - The Hunger Games (The Hunger Games, #1)
     - Inkheart (Inkworld, #1)
     - Harry Potter and the Order of the Phoenix (Harry Potter, #5)
     - Heir of Fire (Throne of Glass, #3)
     - Throne of Glass (Throne of Glass, #1)
     - Bitterblue (Graceling Realm, #3)
     - Crown of Midnight (Throne of Glass, #2)
  Ranking (NCF переранжував, фінальний топ-5):
     * Inkheart (Inkworld, #1)
     * Opal (Lux, #3)
     * Midnight Sun (Twilight, #1.5)
     * Scott Pilgrim, Volume 1: Scott Pilgrim's Precious Little Life
     * Every Day (Every Day, #1)

Користувач idx=1063
  Retrieval (Two-Tower, топ-10 кандидатів):
     - Peace Like a River
     - To Kill a Mockingbird
     - Inkheart (Inkworld, #1)
     - Jane Eyre
     - The Hunger Games Trilogy Boxset (The Hunger Games, #1-3)
     - 

**Питання:** навіщо ділити на два етапи, якщо можна ранжувати NCF одразу всі книги?

NCF точна, але їй треба зібрати користувача з кожною книгою каталогу, це повільно (купа запусків на одну людину щоб побудувати ті пари користувач + книга).
Two-Tower швидка, бо рахує вектори користувача й книг окремо, але є втрати точності бо скалярний добуток векторів не вловлює якихось взаємодій ознак.

Тому швидкий Two-Tower відбирає з великої кількості лише частину кандидатів, а потім точна NCF ранжує лише цю частину.

---
## Завдання 5. Теоретичний блок (письмові відповіді)

Спираючись на лекцію та на те, що Ви щойно побачили на реальних даних, дайте розгорнуті відповіді в markdown-клітинці нижче.

1. **Чому Recall@10 такий низький?** На реальних даних усі моделі цього ДЗ дають скромний Recall@10. Назвіть щонайменше дві причини (підказки: бідні контентні ознаки — лише 12 жанрів; розрідженість; те, що val-лайки не охоплюють усіх книг, які користувач *міг би* вподобати).
2. **Як покращити якість, не змінюючи архітектуру?** Які додаткові ознаки книг і користувачів з Goodbooks можна було б під'єднати? (автор, рік, середній рейтинг, повний набір тегів через TF-IDF, текстові ембединги опису через BERT...)
3. **Diversity.** Якщо користувач любить фентезі, чому не варто показувати йому 10 фентезі-книг підряд? Як технічно підмішати різноманітність?
4. **Freshness / cold start.** Нова книга має 0 оцінок. Який підхід цього ДЗ зможе рекомендувати її одразу, а який — ні? Чому?
5. **Watch time > CTR (з лекції).** Поясніть, чому YouTube оптимізує час перегляду, а не CTR, і як це технічно вшито у weighted logistic regression.


**Відповіді**

1. **Чому Recall@10 такий низький?...**

Усі моделі бачать книгу лише як 12 бінарних жанрів. Цього замало, щоб відрізнити тисячі книг, які мають схожі жанри, але різні тематики чи нюанси.

А ще `val_pos` це книги, які користувач прочитав і високо оцінив. Книги, які користувач не бачив, метрика зараховує як промах, а вони могли б сподобатись.

2. **Як покращити якість, не змінюючи архітектуру?...**

- Додати більше ознак, для цього
спробувати метадані книг із `books.csv`.
- Може ще прогнати назву/опис книги через BERT/Sentence-Transformers і подати вектор як ознаки
- Обробити фічі користувача, врахувати дати, бо смаки змінюються з часом

3. **Diversit**

Користувач уже має фентезі, 10 майже ідентичних книг особливо не дає нової цінності. Якщо вгадали жанр невдало, то всі 10 будуть промахом. А різноманіття від такого трохи страхує.

Щоб підмішати маю такі ідеї:
  - Обмежити частку одного жанру в топ-K.
  - Домішувати частку випадкових/нових кандидатів.

4. **Freshness / cold start**

- Two-Tower і NCF зможуть рекомендувати нову книгу одразу: їхні item-вежі будуються з контентних ознак (жанри/теги/текст). Нова книга з 0 оцінок усе одно має жанровий вектор -> отримує ембединг -> потрапляє в retrieval/ranking.
- VSM теж зможе бо він спирається лише на жанровий вектор книги.
- Matrix Factorization з лекції не зможе, бо там ембединг книги вчиться виключно з взаємодій. Книга без оцінок не має рядка в utility-матриці, її латентний вектор невизначений і рекомендувати її нема як, поки не проставляться перші оцінки.

5.
- 1) **Watch time > CTR (з лекції)**

  - CTR легко "зламати" клікбейтом, бо яскраві елементи назви та прев'ю призводять до кліку, але користувач може закрити відео за кілька секунд. Оптимізуючи CTR, система вчиться приваблювати кліки, а не задовольняти користувача.

  - Час перегляду відображає реальну цінність контенту.
- 2) **Як це вшито у weighted logistic regression.**

  - Замість регресії часу перегляду (він має розкид від секунд до годин, чутливий до викидів і довжини відео) задачу ставлять як бінарну класифікацію клік є / нема, але:
    - позитивним прикладам (клікам) дають вагу = час перегляду в секундах,
    - негативним (без кліку) вагу 1.

  -> Навчені шанси (odds) стають пропорційними очікуваному часу перегляду, тож у топ піднімаються відео, що дають найбільше часу перегляду, а не просто найбільше кліків.
